# 04 Run length, warm-up and replication count

Ticket T4.1.3. **Simulated, benchmark-parameterized line; not plant data.** Draft baseline config (not yet frozen as v1.0).

**Decision: terminating simulation.** The system has a natural start and end: a campaign starts on an empty, cleaned and cleared line and ends when the last batch is packed. The real line also starts that way, so the start-up transient is part of what we want to measure and **no warm-up is deleted** in baseline or scenario runs. A steady-state run (one 5-million-vial batch, no changeovers, always-on calendar) is used below **only as a verification aid**: Welch's method shows how long the empty line takes to fill. That is a sanity check on the pipeline-fill mechanics.

In [1]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go

from ffdt.config import load_config
from ffdt.experiments import reps, runner

cfg = load_config("../configs/baseline_line.yaml")
exp = runner.load_experiment("../configs/experiment.yaml")
SEED = exp["seed"]
exp

{'mode': 'terminating',
 'seed': 20261116,
 'n_reps': 30,
 'ci_level': 0.95,
 'rel_half_width_target': 0.02,
 'primary_metrics': ['good_per_ppt_hour', 'line_oee']}

## 1. Welch plot (steady-state verification model)
10 replications × 168 h. The series is good vials packed per hour, averaged across replications and then smoothed with a centered moving average (window w = 6 h, shrinking at the start; Law 2015, §9.5.1). The rule marks warm-up end as the first hour the moving average comes within 5% of its second-half mean. Confirm it by eye.

In [2]:
ss = reps.steady_state_config(cfg)
HOURS, W = 168, 6
series = np.array([reps.hourly_output(ss, SEED, r, HOURS) for r in range(10)])
ma = reps.welch(series, W)
w_end = reps.warmup_end(ma, tol=0.05)
steady = ma[len(ma) // 2 :].mean()
print(
    f"warm-up ends at hour {w_end}; steady output {steady:,.0f} vials/h "
    f"= {steady / 60:.0f} vpm vs filler nominal 240 vpm ({steady / 60 / 240:.0%})"
)

warm-up ends at hour 3; steady output 12,794 vials/h = 213 vpm vs filler nominal 240 vpm (89%)


In [3]:
# Single series: no legend; the title names it. Reference lines are labeled directly.
fig = go.Figure()
fig.add_scatter(
    x=np.arange(len(ma)),
    y=ma,
    mode="lines",
    line=dict(color="#2a78d6", width=2),
    name="Welch moving average",
    hovertemplate="hour %{x}<br>%{y:,.0f} vials/h<extra></extra>",
)
fig.add_hline(
    y=steady,
    line=dict(color="#6b6b6b", width=1, dash="dot"),
    annotation_text=f"second-half mean {steady:,.0f}",
    annotation_position="bottom right",
)
fig.add_vline(
    x=w_end,
    line=dict(color="#6b6b6b", width=1, dash="dash"),
    annotation_text=f"warm-up ends: hour {w_end}",
    annotation_position="top right",
)
fig.update_layout(
    title="Good vials packed per hour, Welch moving average (w = 6 h, 10 reps)",
    xaxis_title="Hour of run",
    yaxis_title="Vials per hour",
    showlegend=False,
    template="plotly_white",
    height=380,
    margin=dict(l=60, r=20, t=60, b=50),
)
fig.update_yaxes(rangemode="tozero", gridcolor="#eeeeee")
fig.update_xaxes(gridcolor="#eeeeee")
fig

In [4]:
# Table view of the plot (first 12 hours, then every 24 h)
idx = list(range(12)) + list(range(24, len(ma), 24))
pd.DataFrame({"hour": idx, "welch_ma_vials_per_h": ma[idx].round(0)}).set_index("hour").T

hour,0,1,2,3,4,5,6,7,8,9,10,11,24,48,72,96,120,144
welch_ma_vials_per_h,6335.0,11354.0,11957.0,12423.0,12742.0,12858.0,12709.0,13119.0,13094.0,13005.0,12962.0,13008.0,12961.0,12746.0,13480.0,13011.0,12843.0,12851.0


## 2. Replication count from a pilot
Pilot of 20 terminating replications of the full campaign. For each metric, n is the smallest number of replications with t₀.₉₇₅,ₙ₋₁ · s / √n ≤ 2% of the mean, where s is the pilot standard deviation.

In [5]:
pilot = runner.run(cfg, n_reps=20, seed=SEED, n_jobs=-1)
metrics = [
    "good_per_ppt_hour",
    "line_oee",
    "good_per_calendar_hour",
    "makespan_h",
    "mean_lead_time_h",
]
tbl = reps.pilot_table(pilot.summary, metrics, rel=exp["rel_half_width_target"])
tbl.round(4)

,metric,pilot_n,mean,sd,cv,half_width_pilot,rel_half_width_pilot,n_required
0,good_per_ppt_hour,20,8255.9150,287.4377,0.0348,134.5250,0.0163,15
1,line_oee,20,0.5733,0.0200,0.0348,0.0093,0.0163,15
2,good_per_calendar_hour,20,3685.6705,595.5872,0.1616,278.7434,0.0756,254
3,makespan_h,20,165.2700,18.9348,0.1146,8.8618,0.0536,129
4,mean_lead_time_h,20,84.1902,25.8360,0.3069,12.0916,0.1436,907


In [6]:
need = int(tbl.set_index("metric").loc[exp["primary_metrics"], "n_required"].max())
print(f"primary metrics need n = {need}; configs/experiment.yaml uses n_reps = {exp['n_reps']}")
assert exp["n_reps"] >= need, "experiment.yaml n_reps is below the pilot requirement"
assert exp["n_reps"] >= 10

primary metrics need n = 15; configs/experiment.yaml uses n_reps = 30


## 3. Reading
- **Primary metrics** (good vials per PPT hour, line OEE) have a CV of about 3.5% across replications, so about 15 replications reach a ±2% half-width. `n_reps = 30` adds margin, because the pilot's own standard deviation is uncertain. 30 replications take about 9 s in parallel. Paired scenario differences under common random numbers usually need fewer replications than this.
- **Calendar-hour throughput, makespan and lead time** would need hundreds of replications for ±2%. This is the weekend-spill effect (planning/08), plus QA review times with CV 0.8. These metrics are reported with their (wider) CIs and are **not** used for pre-registered decisions.
- **Warm-up:** from an empty line, output reaches steady level within a few hours. Washer, tunnel residence (30 min) and buffers must fill first. In a terminating campaign this start-up is real and is kept in the metrics.